# Integrating Gemini API function calling via OpenAI SDK, implementing web search tools, and configuring forced tool choice.

In [ ]:
!pip install -q ddgs
!pip install -q openai

In [ ]:
from ddgs import DDGS
from openai import OpenAI
from google.colab import userdata
from gradio import ChatInterface
import json

In [ ]:
api_key=userdata.get('GEMINI_API')

In [ ]:
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta",
    api_key=api_key
)

In [ ]:
def search_web(query):
  results = DDGS().text(query)
  return results

In [ ]:
tools = [
    {
        "type": "function",
        "function": {  # <--- You need to nest it inside this "function" key
            "name": "search_web",  # Note: you might also want to fix the typo here to "search_web"
            "description": "Get response according to the user's query by searching from web",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A search query string",
                    },
                },
                "required": ["query"],
            },
        },
    },
]

In [ ]:
response = client.chat.completions.create(
    model = "gemini-2.5-flash",
    messages=[{
        "role": "user",
        "content": "what is the news about japanese prime minister and india"
    }],
    tools=tools,
    max_tokens=300
)
print(response.choices[0].message)

In [ ]:
if (response.choices[0].message.tool_calls):
  for call in response.choices[0].message.tool_calls:
    print('The call is: ', call.function.name)
    if call.function.name == 'search_web':
      answer = search_web(json.loads(call.function.arguments)['query'])
      print('The answer is: ', answer)

TASK--

In [ ]:
!pip install -q ddgs
!pip install -q openai

In [ ]:
from ddgs import DDGS
from openai import OpenAI
from google.colab import userdata
from gradio import ChatInterface
import json

In [ ]:
api_key=userdata.get('GEMINI_API')

In [ ]:
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta",
    api_key=api_key
)

In [ ]:
def add_numbers(num1, num2):
    return num1 + num2

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Adds two numbers together.",
            "parameters": {
                "type": "object",
                "properties": {
                    "num1": {"type": "number", "description": "The first number"},
                    "num2": {"type": "number", "description": "The second number"}
                },
                "required": ["num1", "num2"]
            }
        }
    }
]

In [ ]:
messages = [
    {
        "role": "developer",
        "content": "You are a helpful assistant. Whenever the user asks you to add two numbers, calculate the sum, add 1 to it (increment it), and then provide the arguments to the tool such that the tool's addition yields the incremented value."
    },
    {
        "role": "user",
        "content": "Add 15 and 10."
    }
]

In [ ]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "add_numbers"}} # Forces the tool call
)

In [ ]:
tool_call = response.choices[0].message.tool_calls[0]
print(f"Arguments generated by model: {tool_call.function.arguments}")

In [ ]:
arguments = json.loads(tool_call.function.arguments)
final_result = add_numbers(arguments['num1'], arguments['num2'])

print(f"Final Result: {final_result}")